In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

# Fresh load - untouched raw data
df = pd.read_csv('../data/raw/india_job_market_2025.csv')

print(f"Starting shape: {df.shape}")
# Expected: (97929, 17)

Starting shape: (97929, 17)


In [2]:
# Identify rows missing all 5 critical columns simultaneously
critical_cols = ['minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience', 'tagsAndSkills']
systemic_failure_mask = df[critical_cols].isnull().all(axis=1)

print(f"Rows to drop (systemic failure): {systemic_failure_mask.sum()}")

df = df[~systemic_failure_mask].copy()

print(f"Shape after dropping systemic-failure rows: {df.shape}")
# Expected: (97358, 17)

Rows to drop (systemic failure): 571
Shape after dropping systemic-failure rows: (97358, 17)


In [3]:
full_dupes_count = df.duplicated().sum()
print(f"Full duplicate rows to drop: {full_dupes_count}")

df = df.drop_duplicates(keep='first').copy()

print(f"Shape after dropping full duplicates: {df.shape}")
# Expected: (97111, 17)

Full duplicate rows to drop: 247
Shape after dropping full duplicates: (97111, 17)


In [4]:
# Drop rows where BOTH jobId AND title match (true duplicates)
# This correctly spares the 11025020549 pair since their titles differ
true_dupe_mask = df.duplicated(subset=['jobId', 'title'], keep='first')

print(f"True partial duplicates to drop: {true_dupe_mask.sum()}")

df = df[~true_dupe_mask].copy()

print(f"Shape after resolving partial duplicates: {df.shape}")
# Expected: (97109, 17)

True partial duplicates to drop: 2
Shape after resolving partial duplicates: (97109, 17)


In [5]:
missing_company = df['companyName'].isnull().sum()
print(f"Rows missing companyName: {missing_company}")

df = df.dropna(subset=['companyName']).copy()

print(f"Shape after dropping missing companyName: {df.shape}")
# Expected: (97105, 17)

Rows missing companyName: 4
Shape after dropping missing companyName: (97105, 17)


In [6]:
print("=" * 60)
print("PHASE 5 - CHECKPOINT 1 SUMMARY")
print("=" * 60)
print(f"Original rows:        97,929")
print(f"Current rows:          {len(df):,}")
print(f"Total removed:         {97929 - len(df):,}")
print(f"Removal percentage:    {(97929 - len(df)) / 97929 * 100:.2f}%")
print(f"\nRemaining null check:")
print(df[critical_cols].isnull().sum())

PHASE 5 - CHECKPOINT 1 SUMMARY
Original rows:        97,929
Current rows:          97,105
Total removed:         824
Removal percentage:    0.84%

Remaining null check:
minimumSalary        0
maximumSalary        0
minimumExperience    0
maximumExperience    0
tagsAndSkills        0
dtype: int64


In [7]:
import re

def extract_work_mode(location_str):
    """
    Extracts work mode (Remote/Hybrid/Onsite) from the location string.
    Returns the work mode and does NOT modify the original string.
    """
    location_lower = str(location_str).lower()
    
    if 'remote' in location_lower:
        return 'Remote'
    elif 'hybrid' in location_lower:
        return 'Hybrid'
    else:
        return 'Onsite'

df['work_mode'] = df['location'].apply(extract_work_mode)

print(df['work_mode'].value_counts())
print(f"\nTotal: {df['work_mode'].value_counts().sum()}")

work_mode
Onsite    89452
Hybrid     5733
Remote     1920
Name: count, dtype: int64

Total: 97105


In [8]:
# Rows that were originally exactly "Remote" but did NOT get classified as Remote now
exactly_remote = df[df['location'].str.strip().str.lower() == 'remote']
print(f"Rows where location is exactly 'Remote': {len(exactly_remote)}")

not_classified_remote = exactly_remote[exactly_remote['work_mode'] != 'Remote']
print(f"Of those, how many were NOT classified as Remote: {len(not_classified_remote)}")
print(not_classified_remote[['location', 'work_mode']].head())

Rows where location is exactly 'Remote': 1914
Of those, how many were NOT classified as Remote: 0
Empty DataFrame
Columns: [location, work_mode]
Index: []


In [9]:
def extract_primary_city(location_str):
    """
    Extracts the primary (first-listed) city from a messy location string.
    Order of operations matters: strip work-mode prefix -> strip parentheses -> split on comma -> trim.
    """
    text = str(location_str)
    
    # Step 1: Remove work-mode prefixes like "Hybrid - " or "Remote - "
    text = re.sub(r'^(Hybrid|Remote)\s*-\s*', '', text, flags=re.IGNORECASE)
    
    # Step 2: Remove anything inside parentheses, e.g. "(Motera)" or "(Hennur +2)"
    text = re.sub(r'\(.*?\)', '', text)
    
    # Step 3: Split on comma, take the first part (primary city)
    primary = text.split(',')[0]
    
    # Step 4: Trim whitespace
    primary = primary.strip()
    
    return primary

df['primary_city'] = df['location'].apply(extract_primary_city)

print(f"Unique cities BEFORE standardization: {df['primary_city'].nunique()}")
print(f"\nTop 20 cities:")
print(df['primary_city'].value_counts().head(20))

Unique cities BEFORE standardization: 1305

Top 20 cities:
primary_city
Bengaluru          19292
Hyderabad          12308
Pune                9337
Mumbai              6591
Chennai             6398
Gurugram            5187
Noida               4930
Ahmedabad           2408
Kolkata             1956
Remote              1914
Navi Mumbai         1696
Thane               1027
Kochi                976
Jaipur               960
Mumbai Suburban      781
Coimbatore           746
Chandigarh           621
Mohali               615
Vadodara             588
Nagpur               576
Name: count, dtype: int64


In [10]:
# Check specifically for known spelling variants
known_variants = ['Bangalore', 'Bengaluru', 'Gurgaon', 'Gurugram', 'Cochin', 'Kochi', 'Pondicherry', 'Puducherry']
variant_check = df['primary_city'].value_counts()
for v in known_variants:
    if v in variant_check.index:
        print(f"{v:15s} : {variant_check[v]}")

Bengaluru       : 19292
Gurugram        : 5187
Kochi           : 976
Pondicherry     : 1
Puducherry      : 25


In [11]:
# See the bottom of the distribution - this reveals junk/rare entries
print(df['primary_city'].value_counts().tail(40))

primary_city
Chhipabarod     1
Garhshankar     1
Chitapur        1
Borio           1
Bijbehara       1
Indri           1
Lodhika         1
Azara           1
Seppa           1
Sattenapalle    1
Bathalapalle    1
Atreyapuram     1
Yellapur        1
Halvad          1
Dhemaji         1
Dongargaon      1
Kadegaon        1
Nawabganj       1
Manjhanpur      1
Janjgir         1
Tirodi          1
Singapore       1
Indora          1
Odisha          1
Moinabad        1
Berhampore      1
Baska           1
Muktsar         1
Nabadwip        1
Akbarpur        1
Beri            1
Hajipur         1
Islampur        1
Surajpur        1
Madhepura       1
Nevasa          1
Kawardha        1
Bokakhat        1
Bamanwas        1
Tumsar          1
Name: count, dtype: int64


In [12]:
# How many cities have fewer than 5 postings? (likely junk/noise/rare typos)
rare_cities = df['primary_city'].value_counts()
print(f"Cities with fewer than 5 postings: {(rare_cities < 5).sum()}")
print(f"Total rows affected by these rare cities: {rare_cities[rare_cities < 5].sum()}")

Cities with fewer than 5 postings: 839
Total rows affected by these rare cities: 1439


In [13]:
# --- 1. Fix spelling variant ---
df['primary_city'] = df['primary_city'].replace({'Pondicherry': 'Puducherry'})

# --- 2. Fix "Remote" leaking into city field ---
df.loc[df['primary_city'] == 'Remote', 'primary_city'] = 'Not Specified'

# --- 3. Flag the one non-India entry ---
df.loc[df['primary_city'] == 'Singapore', 'primary_city'] = 'International/Remote'

# --- 4. Flag the state-name-as-city anomaly ---
df.loc[df['primary_city'] == 'Odisha', 'primary_city'] = 'Unspecified'

# --- 5. Build city_tier_group for clean visual rollups ---
city_counts = df['primary_city'].value_counts()
rare_cities = city_counts[city_counts < 5].index

def assign_city_tier_group(city):
    if city in ['Not Specified', 'International/Remote', 'Unspecified']:
        return city
    elif city in rare_cities:
        return 'Other (Tier 3/4 Town)'
    else:
        return city

df['city_tier_group'] = df['primary_city'].apply(assign_city_tier_group)

# --- 6. Build metro_region rollup for major metros ---
metro_map = {
    'Mumbai': 'Mumbai Metro', 'Navi Mumbai': 'Mumbai Metro', 
    'Thane': 'Mumbai Metro', 'Mumbai Suburban': 'Mumbai Metro',
    'Gurugram': 'Delhi NCR', 'Noida': 'Delhi NCR', 
    'New Delhi': 'Delhi NCR', 'Delhi': 'Delhi NCR', 'Faridabad': 'Delhi NCR', 'Ghaziabad': 'Delhi NCR'
}

df['metro_region'] = df['primary_city'].map(metro_map).fillna(df['primary_city'])

# --- Verification ---
print(f"Unique primary_city values: {df['primary_city'].nunique()}")
print(f"Unique city_tier_group values: {df['city_tier_group'].nunique()}")
print(f"\nTop 15 city_tier_group:")
print(df['city_tier_group'].value_counts().head(15))
print(f"\nTop 10 metro_region:")
print(df['metro_region'].value_counts().head(10))

Unique primary_city values: 1304
Unique city_tier_group values: 469

Top 15 city_tier_group:
city_tier_group
Bengaluru                19292
Hyderabad                12308
Pune                      9337
Mumbai                    6591
Chennai                   6398
Gurugram                  5187
Noida                     4930
Ahmedabad                 2408
Kolkata                   1956
Not Specified             1914
Navi Mumbai               1696
Other (Tier 3/4 Town)     1436
Thane                     1027
Kochi                      976
Jaipur                     960
Name: count, dtype: int64

Top 10 metro_region:
metro_region
Bengaluru        19292
Hyderabad        12308
Delhi NCR        11283
Mumbai Metro     10095
Pune              9337
Chennai           6398
Ahmedabad         2408
Kolkata           1956
Not Specified     1914
Kochi              976
Name: count, dtype: int64


In [14]:
# Confirm the math
total_unique = df['primary_city'].nunique()
rare_count = (df['primary_city'].value_counts() < 5).sum()
non_rare_count = total_unique - rare_count

print(f"Total unique cities: {total_unique}")
print(f"Rare cities (<5 postings): {rare_count}")
print(f"Non-rare cities (kept as-is): {non_rare_count}")
print(f"Expected city_tier_group count: {non_rare_count} + 1 (Other bucket) + flags already inside non_rare or separate")

Total unique cities: 1304
Rare cities (<5 postings): 838
Non-rare cities (kept as-is): 466
Expected city_tier_group count: 466 + 1 (Other bucket) + flags already inside non_rare or separate


In [15]:
# Better strategy for a DASHBOARD-ready grouping: Top 25 cities explicitly, everything else -> Other
top_25_cities = df['primary_city'].value_counts().head(25).index.tolist()

def assign_city_tier_group_v2(city):
    if city in ['Not Specified', 'International/Remote', 'Unspecified']:
        return city
    elif city in top_25_cities:
        return city
    else:
        return 'Other Cities'

df['city_tier_group'] = df['primary_city'].apply(assign_city_tier_group_v2)

print(f"Unique city_tier_group values: {df['city_tier_group'].nunique()}")
print(f"\nFull distribution:")
print(df['city_tier_group'].value_counts())

Unique city_tier_group values: 28

Full distribution:
city_tier_group
Bengaluru               19292
Other Cities            15936
Hyderabad               12308
Pune                     9337
Mumbai                   6591
Chennai                  6398
Gurugram                 5187
Noida                    4930
Ahmedabad                2408
Kolkata                  1956
Not Specified            1914
Navi Mumbai              1696
Thane                    1027
Kochi                     976
Jaipur                    960
Mumbai Suburban           781
Coimbatore                746
Chandigarh                621
Mohali                    615
Vadodara                  588
Nagpur                    576
Surat                     499
Lucknow                   473
Nashik                    460
New Delhi                 454
Faridabad                 374
International/Remote        1
Unspecified                 1
Name: count, dtype: int64


In [16]:
print(df['title'].value_counts().head(30))

title
Application Developer             2033
Application Lead                  1346
Sales Executive                    576
Software Development Engineer      497
Business Development Executive     455
Trust & Safety New Associate       348
Data Engineer                      338
Business Development Manager       323
Sales Manager                      319
Java Full Stack Developer          282
Java Developer                     253
Relationship Manager               232
Area Sales Manager                 217
Application Support Engineer       217
Accountant                         215
Customer Support Executive         213
Senior Software Engineer           206
Account Executive                  205
Graphic Designer                   188
Software Development Lead          181
Business Development Associate     177
Business Analyst                   172
Technical Lead - L1                165
Software Engineer                  164
AI / ML Engineer                   153
Sales Officer Home 

In [17]:
# Are there any obvious non-tech roles still in here that we flagged back in Phase 4?
non_tech_keywords = ['Sales', 'HR', 'Recruiter', 'Marketing', 'Accountant', 'Teacher', 'Nurse', 'Driver']
for kw in non_tech_keywords:
    count = df['title'].str.contains(kw, case=False, na=False).sum()
    print(f"{kw:12s}: {count}")

Sales       : 9181
HR          : 1613
Recruiter   : 935
Marketing   : 1745
Accountant  : 748
Teacher     : 454
Nurse       : 189
Driver      : 70


In [18]:
import re

# Keywords that need WORD BOUNDARY matching (short/ambiguous words that cause false matches as substrings)
# Keywords NOT in this list will still use substring matching (safe for longer, specific phrases)

tech_keywords = [
    'Developer', 'Engineer', 'Programmer', 'Architect', 'DevOps',
    'Data Scientist', 'Data Analyst', 'Data Engineer', 'Analytics',
    'Software', 'Application', 'Full Stack', 'Fullstack', 'Frontend', 'Backend',
    'Java', 'Python', 'SQL', 'Cloud', 'AWS', 'Azure', r'\bAI\b', r'\bML\b',
    'Machine Learning', r'\bQA\b', 'Test Engineer', 'SDET', 'Network Engineer',
    'System Administrator', 'Database Administrator', r'\bDBA\b', 'Cybersecurity',
    'Security Engineer', 'Security Architect', 'Business Analyst', 'Business Architect',
    r'\bIT\b', 'Technical Lead', 'Technical Project Manager', 'Technical Architect',
    'Web Developer', 'Mobile Developer', 'UI/UX', 'Solution Architect',
    'Automation', 'Embedded', 'Firmware', 'Blockchain', 'Quality Analyst',
    'Quality Engineer'
]

exclude_keywords = [
    r'\bSales\b',  # word boundary - won't match "Salesforce"
    'Business Development', 'Marketing', r'\bHR\b', 'Human Resource',
    'Recruiter', 'Recruitment', 'Accountant', 'Accounting', 'Finance Officer',
    'Financial Analyst', 'Teacher', 'Professor', 'Nurse', 'Doctor', 'Driver',
    'Customer Support', 'Customer Care', 'Relationship Manager', 'Account Executive',
    'Field Sales', 'Telecaller', 'Insurance', 'Loan', 'Banking Officer',
    'Operations Executive', 'Logistics', 'Warehouse', 'Retail', 'Store Manager',
    'Hospitality', 'Chef', 'Housekeeping', 'Security Guard',
    'Record To Report', 'Order To Cash', 'Procure To Pay', 'Hire To Retire',
    'Technical Recruiter', 'Sales Engineer'
]

def is_tech_role_v2(title):
    title_str = str(title)
    
    # Check excludes first (using regex for word-boundary terms, substring for phrases)
    for kw in exclude_keywords:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return False
        else:
            if kw.lower() in title_str.lower():
                return False
    
    # Check includes
    for kw in tech_keywords:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return True
        else:
            if kw.lower() in title_str.lower():
                return True
    
    return False

df['is_tech_role'] = df['title'].apply(is_tech_role_v2)

print(f"Tech roles identified: {df['is_tech_role'].sum():,}")
print(f"Non-tech roles identified: {(~df['is_tech_role']).sum():,}")
print(f"Tech role percentage: {df['is_tech_role'].mean()*100:.2f}%")

Tech roles identified: 32,283
Non-tech roles identified: 64,822
Tech role percentage: 33.25%


In [19]:
# See what got classified as tech - spot check the top 30
print("TOP 30 TECH-CLASSIFIED TITLES:")
print(df[df['is_tech_role']]['title'].value_counts().head(30))

TOP 30 TECH-CLASSIFIED TITLES:
title
Application Developer                    2033
Application Lead                         1346
Software Development Engineer             497
Data Engineer                             338
Java Full Stack Developer                 282
Java Developer                            253
Application Support Engineer              217
Senior Software Engineer                  206
Software Development Lead                 181
Business Analyst                          172
Technical Lead - L1                       165
Software Engineer                         164
AI / ML Engineer                          153
Developer - L3                            142
Application Designer                      118
Senior Data Engineer                      116
Application Tech Support Practitioner     115
Full Stack Engineer                       112
Data Scientist                            108
Security Architect                        102
Solution Architect                        1

In [20]:
# See what got classified as non-tech but CONTAINS a tech-sounding word - catch false negatives
tech_sounding_but_excluded = df[~df['is_tech_role']]['title'].str.contains(
    'Engineer|Developer|Analyst|Technical', case=False, na=False
)
print(f"\nPotential false negatives (tech-sounding but excluded): {tech_sounding_but_excluded.sum()}")
print(df[~df['is_tech_role']][tech_sounding_but_excluded]['title'].value_counts().head(20))


Potential false negatives (tech-sounding but excluded): 4150
title
Analyst                                        84
Financial Analyst                              68
Sales Engineer                                 66
Record To Report Ops Analyst                   35
Senior Analyst                                 33
Delivery Operations Senior Analyst             33
Analyst - KYC                                  33
Sales Executive / Graduate Engineer Trainee    28
Technical Writer                               22
Technical Recruiter                            22
Business Interlock Analyst                     22
Record To Report Ops Senior Analyst            22
I&F Decision Sci Practitioner Sr Analyst       21
Order To Cash Operations Analyst               18
Business Interlock Senior Analyst              17
Trust & Safety Analyst                         15
Procure To Pay Operations Senior Analyst       14
Cyber Security Analyst - L4                    14
HR Service Delivery Analyst     

In [21]:
print(f"Tech roles identified: {df['is_tech_role'].sum():,}")
print(f"Tech role percentage: {df['is_tech_role'].mean()*100:.2f}%")

# Verify Salesforce Developer is now correctly captured
print("\nSalesforce Developer check:")
print(df[df['title'].str.contains('Salesforce', case=False, na=False)]['is_tech_role'].value_counts())

# Verify Technical Project Manager is now correctly captured
print("\nTechnical Project Manager check:")
print(df[df['title'].str.contains('Technical Project Manager', case=False, na=False)]['is_tech_role'].value_counts())

# Re-run the false negative check to see what's left
tech_sounding_but_excluded = df[~df['is_tech_role']]['title'].str.contains(
    'Engineer|Developer|Analyst|Technical', case=False, na=False
)
print(f"\nRemaining potential false negatives: {tech_sounding_but_excluded.sum()}")
print(df[~df['is_tech_role']][tech_sounding_but_excluded]['title'].value_counts().head(15))

Tech roles identified: 32,283
Tech role percentage: 33.25%

Salesforce Developer check:
is_tech_role
True     237
False    119
Name: count, dtype: int64

Technical Project Manager check:
is_tech_role
True    69
Name: count, dtype: int64

Remaining potential false negatives: 4150
title
Analyst                                        84
Financial Analyst                              68
Sales Engineer                                 66
Record To Report Ops Analyst                   35
Senior Analyst                                 33
Delivery Operations Senior Analyst             33
Analyst - KYC                                  33
Sales Executive / Graduate Engineer Trainee    28
Technical Writer                               22
Technical Recruiter                            22
Business Interlock Analyst                     22
Record To Report Ops Senior Analyst            22
I&F Decision Sci Practitioner Sr Analyst       21
Order To Cash Operations Analyst               18
Business Inter

In [22]:
print(df[df['title'].str.contains('Salesforce', case=False, na=False) & (~df['is_tech_role'])]['title'].value_counts().head(15))

title
Salesforce Administrator                                   9
Salesforce Marketing Cloud Developer                       5
Package Consultant-Salesforce                              5
Salesforce Lead                                            4
Senior Salesforce Consultant                               3
Salesforce Consultant                                      3
Salesforce Nonprofit, Sr Associate                         2
Salesforce Leader                                          2
Salesforce Marketing Cloud Professional                    2
Consultant, Salesforce development                         2
AD/SM Program Project Management Salesforce                2
Salesforce lead                                            1
Salesforce CPQ Administrator                               1
Salesforce Vlocity/Omnistudio | PAN India | 6-10 yrs !!    1
Senior Salesforce Administrator                            1
Name: count, dtype: int64


In [23]:
# Add missing tech-specific terms
tech_keywords_v3 = tech_keywords + [
    'Cyber Security', 'Security Analyst', 'Cloud Security', 
    'Information Security', 'SOC Analyst', 'Penetration Test'
]

def is_tech_role_v3(title):
    title_str = str(title)
    
    for kw in exclude_keywords:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return False
        else:
            if kw.lower() in title_str.lower():
                return False
    
    for kw in tech_keywords_v3:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return True
        else:
            if kw.lower() in title_str.lower():
                return True
    
    return False

df['is_tech_role'] = df['title'].apply(is_tech_role_v3)

print(f"Tech roles identified: {df['is_tech_role'].sum():,}")
print(f"Tech role percentage: {df['is_tech_role'].mean()*100:.2f}%")

# Verify Cyber Security Analyst now correctly captured
print("\nCyber Security Analyst check:")
print(df[df['title'].str.contains('Cyber Security Analyst', case=False, na=False)]['is_tech_role'].value_counts())

Tech roles identified: 32,438
Tech role percentage: 33.41%

Cyber Security Analyst check:
is_tech_role
True    27
Name: count, dtype: int64


In [24]:
# Create the tech-only dataset
df_tech = df[df['is_tech_role'] == True].copy()

print(f"Full cleaned dataset: {len(df):,} rows")
print(f"Tech-only dataset: {len(df_tech):,} rows")
print(f"Tech-only percentage: {len(df_tech)/len(df)*100:.2f}%")

# Reset index for cleanliness going forward
df_tech = df_tech.reset_index(drop=True)

print(f"\nFinal df_tech shape: {df_tech.shape}")

Full cleaned dataset: 97,105 rows
Tech-only dataset: 32,438 rows
Tech-only percentage: 33.41%

Final df_tech shape: (32438, 22)


In [25]:
print(df_tech.columns.tolist())

['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience', 'work_mode', 'primary_city', 'city_tier_group', 'metro_region', 'is_tech_role']


In [28]:
def assign_role_category(title):
    t = str(title).lower()
    
    # Order matters — more specific patterns checked before generic ones
    
    # ML / AI first (before "Engineer" catches it generically)
    if any(kw in t for kw in ['machine learning', 'ml engineer', 'ai engineer', 
                               'ai/ml', 'ai / ml', 'nlp', 'computer vision', 
                               'llm', 'generative ai', 'deep learning']):
        return 'ML / AI Engineer'
    
    # Data Science
    if any(kw in t for kw in ['data scientist', 'data science']):
        return 'Data Scientist'
    
    # Data Engineering
    if any(kw in t for kw in ['data engineer', 'etl', 'big data', 
                               'data pipeline', 'spark', 'hadoop', 'databricks']):
        return 'Data Engineer'
    
    # Data Analyst / BI
    if any(kw in t for kw in ['data analyst', 'business intelligence', 
                               'bi analyst', 'bi developer', 'analytics engineer',
                               'reporting analyst']):
        return 'Data Analyst'
    
    # Security
    if any(kw in t for kw in ['security', 'cybersecurity', 'cyber security', 
                               'soc analyst', 'penetration', 'vapt', 'infosec']):
        return 'Security Engineer'
    
    # DevOps / Cloud
    if any(kw in t for kw in ['devops', 'cloud engineer', 'sre', 'site reliability',
                               'platform engineer', 'infrastructure engineer',
                               'aws engineer', 'azure engineer', 'kubernetes', 
                               'docker engineer']):
        return 'DevOps / Cloud Engineer'
    
    # QA / Testing
    if any(kw in t for kw in ['quality engineer', 'quality analyst', 'qa ', 'qe ',
                               'test engineer', 'sdet', 'automation tester',
                               'manual tester', 'testing']):
        return 'QA / Test Engineer'
    
    # Frontend
    if any(kw in t for kw in ['frontend', 'front end', 'front-end', 'ui developer',
                               'react developer', 'angular developer', 
                               'vue developer', 'ui engineer']):
        return 'Frontend Developer'
    
    # Mobile
    if any(kw in t for kw in ['android', 'ios developer', 'flutter', 
                               'react native', 'mobile developer', 'mobile engineer']):
        return 'Mobile Developer'
    
    # Backend
    if any(kw in t for kw in ['backend', 'back end', 'back-end', 'node developer',
                               'django developer', 'spring boot', 'api developer']):
        return 'Backend Developer'
    
    # Full Stack (before generic Software Engineer)
    if any(kw in t for kw in ['full stack', 'fullstack', 'full-stack', 
                               'mern', 'mean stack', 'lamp stack']):
        return 'Full Stack Developer'
    
    # Technical Lead / Architect (before generic Software Engineer)
    if any(kw in t for kw in ['technical lead', 'tech lead', 'solution architect',
                               'enterprise architect', 'technology architect',
                               'application architect', 'it architect']):
        return 'Technical Lead / Architect'
    
    # Database
    if any(kw in t for kw in ['dba', 'database administrator', 'sql server dba',
                               'oracle dba', 'database engineer', 'mysql dba']):
        return 'Database Administrator'
    
    # IT Support / Sysadmin
    if any(kw in t for kw in ['system administrator', 'sysadmin', 'it support',
                               'network engineer', 'network administrator',
                               'helpdesk', 'help desk', 'desktop support',
                               'l1 support', 'l2 support', 'l3 support']):
        return 'IT Support / Sysadmin'
    
    # Product Manager
    if any(kw in t for kw in ['product manager', 'product owner', 
                               'program manager', 'technical program manager']):
        return 'Product Manager'
    
    # Business Analyst (before generic Software Engineer)
    if any(kw in t for kw in ['business analyst', 'business architect',
                               'systems analyst', 'functional consultant',
                               'business systems', 'process analyst']):
        return 'Business Analyst'
    
    # Generic Software Engineer — broad catch after all specific ones above
    if any(kw in t for kw in ['software engineer', 'software developer',
                               'application developer', 'application lead',
                               'application designer', 'application support',
                               'developer', 'engineer', 'programmer',
                               'java', 'python developer', 'dot net', '.net',
                               'php', 'scala', 'golang', 'rust developer',
                               'c++ developer', 'embedded']):
        return 'Software Engineer'
    
    return 'Other Tech'

df_tech['role_category'] = df_tech['title'].apply(assign_role_category)

print(df_tech['role_category'].value_counts())
print(f"\nTotal categories used: {df_tech['role_category'].nunique()}")
print(f"Rows in 'Other Tech': {(df_tech['role_category'] == 'Other Tech').sum()}")
print(f"Other Tech percentage: {(df_tech['role_category'] == 'Other Tech').mean()*100:.2f}%")

role_category
Software Engineer             17503
Other Tech                     3587
QA / Test Engineer             1712
Full Stack Developer           1706
Data Engineer                  1300
DevOps / Cloud Engineer        1070
Technical Lead / Architect     1065
Security Engineer               623
ML / AI Engineer                613
Business Analyst                594
Backend Developer               559
Frontend Developer              441
IT Support / Sysadmin           368
Data Analyst                    367
Mobile Developer                352
Data Scientist                  333
Database Administrator          197
Product Manager                  48
Name: count, dtype: int64

Total categories used: 18
Rows in 'Other Tech': 3587
Other Tech percentage: 11.06%


In [29]:
print(df_tech[df_tech['role_category'] == 'Other Tech']['title'].value_counts().head(20))

title
Software Development Lead                                         181
Application Tech Support Practitioner                             115
Architect                                                          69
Data Architect                                                     48
Business Process Architect                                         42
Team Leader - Software                                             36
Technical Architect                                                30
Technical Project Manager                                          28
Large Language Model Architect                                     27
Senior Architect                                                   26
Test Automation Lead                                               20
UI/UX Designer                                                     19
IT Executive                                                       18
Package Consultant-Oracle Finance Cloud                            16
Senior Data Ar